# License Plate Validator

## Purpose

This notebook implements and demonstrates a configurable license plate validator. Given a `.cnf` JSON config file and a license plate string, the module checks whether the plate satisfies the constraints defined in the config.

## Design Decisions

- **Config-driven validation**: constraints are not hardcoded. The same validation logic works for any plate format by swapping the config file.
- **Partial constraints**: not all three keys (`length`, `letters`, `numbers`) are required. Only the ones present in the config are enforced, making the module flexible for formats that only care about, say, total length.
- **Isolated config loading**: `load_config` is a separate function from `validate_plate`. This separation makes each function independently testable and keeps responsibilities clear.
- **No third-party dependencies**: the module relies only on Python's standard library (`json`, `os`), so it works out of the box.

In [1]:
"""License plate validator module."""

import json
import os


def load_config(config_filename: str) -> dict:
    """Load and return the validation config from a JSON .cnf file.

    The config file is resolved relative to the current working directory.
    It may contain any combination of the following keys:
        - length (int): expected total length of the license plate
        - letters (int): expected number of letter characters
        - numbers (int): expected number of digit characters

    Args:
        config_filename: Name of the .cnf JSON file (e.g. 'rules.cnf').

    Returns:
        A dict with the parsed config. Missing keys are simply absent.

    Raises:
        FileNotFoundError: If the config file does not exist.
        ValueError: If the file content is not valid JSON.
    """
    config_path = os.path.join(os.getcwd(), config_filename)

    try:
        with open(config_path, "r", encoding="utf-8") as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON in config file '{config_filename}': {e}") from e


def validate_plate(plate: str, config_filename: str) -> bool:
    """Validate a license plate string against the rules defined in a config file.

    Only the constraints present in the config are enforced. For example, if
    the config only defines 'length', the character-type counts are not checked.

    Args:
        plate: The license plate string to validate.
        config_filename: Name of the .cnf JSON config file to load rules from.

    Returns:
        True if the plate satisfies all defined constraints, False otherwise.
    """
    config = load_config(config_filename)

    if "length" in config and len(plate) != config["length"]:
        return False

    if "letters" in config and sum(c.isalpha() for c in plate) != config["letters"]:
        return False

    if "numbers" in config and sum(c.isdigit() for c in plate) != config["numbers"]:
        return False

    return True

In [2]:
import json

configs = {
    "brasil_old.cnf": {"length": 7, "letters": 3, "numbers": 4},
    "brasil_mercosul.cnf": {"length": 7, "letters": 4, "numbers": 3},
}

for filename, content in configs.items():
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(content, f, indent=2)
    print(f"Written: {filename} -> {content}")

Written: brasil_old.cnf -> {'length': 7, 'letters': 3, 'numbers': 4}
Written: brasil_mercosul.cnf -> {'length': 7, 'letters': 4, 'numbers': 3}


In [3]:
import unittest


class TestLicensePlateValidator(unittest.TestCase):

    def test_valid_old_format(self):
        # A classic Brazilian plate (3 letters + 4 digits) should pass brasil_old.cnf
        self.assertTrue(validate_plate("ABC1234", "brasil_old.cnf"))

    def test_valid_mercosul_format(self):
        # A Mercosul plate (4 letters + 3 digits, mixed) should pass brasil_mercosul.cnf
        self.assertTrue(validate_plate("ABC1D23", "brasil_mercosul.cnf"))

    def test_fails_length_constraint(self):
        # A plate with fewer than 7 characters should fail the length constraint
        self.assertFalse(validate_plate("ABC123", "brasil_old.cnf"))

    def test_fails_letters_constraint(self):
        # A 7-char plate with only 2 letters should fail the letters constraint (expects 3)
        self.assertFalse(validate_plate("AB12345", "brasil_old.cnf"))

    def test_fails_numbers_constraint(self):
        # A 7-char plate with only 3 digits should fail the numbers constraint (expects 4)
        self.assertFalse(validate_plate("ABCD123", "brasil_old.cnf"))

    def test_fails_multiple_constraints(self):
        # A short all-letter plate violates length, numbers, and letters simultaneously
        self.assertFalse(validate_plate("ABC", "brasil_old.cnf"))

    def test_config_file_not_found(self):
        # A non-existent config file should raise FileNotFoundError
        with self.assertRaises(FileNotFoundError):
            validate_plate("ABC1234", "nonexistent.cnf")


# Run the suite and display results inline
loader = unittest.TestLoader()
suite = loader.loadTestsFromTestCase(TestLicensePlateValidator)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

test_config_file_not_found (__main__.TestLicensePlateValidator.test_config_file_not_found) ... ok
test_fails_length_constraint (__main__.TestLicensePlateValidator.test_fails_length_constraint) ... ok
test_fails_letters_constraint (__main__.TestLicensePlateValidator.test_fails_letters_constraint) ... ok
test_fails_multiple_constraints (__main__.TestLicensePlateValidator.test_fails_multiple_constraints) ... ok
test_fails_numbers_constraint (__main__.TestLicensePlateValidator.test_fails_numbers_constraint) ... ok
test_valid_mercosul_format (__main__.TestLicensePlateValidator.test_valid_mercosul_format) ... ok
test_valid_old_format (__main__.TestLicensePlateValidator.test_valid_old_format) ... ok

----------------------------------------------------------------------
Ran 7 tests in 0.007s

OK


<unittest.runner.TextTestResult run=7 errors=0 failures=0>

## Assumptions

- **Config file location**: in the standalone module (`license_plate_validator.py`) the config is resolved relative to the module file itself. Inside this notebook, it is resolved relative to `os.getcwd()`, which is the directory where the notebook is running. Both `.cnf` files are written to that same directory in the cell above, so the notebook is fully self-contained.
- **Character classification**: `str.isalpha()` and `str.isdigit()` are used to count letters and digits respectively. Any character that is neither (e.g. hyphens, spaces) is silently ignored by the counts — it would only affect the `length` constraint.
- **Case insensitivity**: the validator treats uppercase and lowercase letters equally, since `isalpha()` returns `True` for both. No normalisation step is applied.
- **Flat config structure**: the `.cnf` file is expected to be a flat JSON object. Nested structures are not supported and would be ignored.
- **All constraints are integers**: the values for `length`, `letters`, and `numbers` are assumed to be non-negative integers. No type validation is performed on the config values themselves.